# Ollama check + pull models

In [1]:
import requests, subprocess, json, re
from datetime import datetime, timedelta

def ollama_is_up(base_url="http://localhost:11434"):
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

assert ollama_is_up(), "Ollama is not responding at http://localhost:11434. Start Ollama and try again."

def ollama_pull(model: str):
    print(f"Pulling model: {model}")
    subprocess.run(["ollama", "pull", model], check=True)

# Required
ollama_pull("phi4")

# Optional medical LLM (free). Uncomment if you want to test a medical-tuned model.
# ollama_pull("meditron")

Pulling model: phi4


# LLM + advanced prompts (Patient Agent + Supervisor Agent)

In [2]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

MODEL = "phi4"        # try: "meditron"
TEMPERATURE = 0.2     # lower => more deterministic
BASE_URL = "http://localhost:11434"

llm = ChatOllama(model=MODEL, temperature=TEMPERATURE, base_url=BASE_URL)

PATIENT_AGENT_SYSTEM = """\
You are a cardiovascular care assistant agent focused on heart failure and post-myocardial infarction follow-up.
Goal: collect daily symptoms/vitals, support adherence, provide safe guidance, and propose next actions.

CRITICAL RULES:
- You DO NOT prescribe, DO NOT change medication doses, DO NOT tell the patient to stop medications.
- You DO NOT provide a definitive diagnosis.
- Use concise, clear language.
- Output MUST be ONLY valid JSON matching the schema below.

JSON SCHEMA:
{
  "summary": "what the patient reported",
  "risk_level": "stable|watch|urgent",
  "rationale": ["reason 1", "reason 2"],
  "questions": ["short question 1", "short question 2"],
  "patient_instructions": [
    "safe instruction 1",
    "safe instruction 2"
  ],
  "proposed_actions": [
    {"type":"schedule|message_team|self_care|none", "payload":{...}}
  ]
}

CLINICAL HEURISTICS (do not mention guidelines explicitly):
- HF worsening: progressive dyspnea, rapid weight gain, edema, orthopnea, marked fatigue.
- Urgent red flags: significant chest pain, severe dyspnea/rest dyspnea, syncope, acute neuro symptoms.
"""

SUPERVISOR_SYSTEM = """\
You are the Clinical Safety Supervisor Agent. Your job is to review the Patient Agent output and enforce safety.

You must:
1) detect unsafe content (medication changes, dosing advice, definitive diagnosis)
2) verify risk_level consistency with patient data
3) enforce escalation policies
4) output ONLY valid JSON

JSON FORMAT:
{
  "decision": "approve|revise|escalate_routine|escalate_urgent|block",
  "issues": ["issue 1", "issue 2"],
  "safe_revision": { ...revised Patient Agent JSON... }  // required if decision="revise"
}

POLICIES:
- If red flags (significant chest pain, severe dyspnea/rest dyspnea, syncope, acute neuro symptoms): escalate_urgent.
- If moderate worsening (weight gain + edema + mild/moderate dyspnea): escalate_routine.
- If any medication/dose change advice: revise or block.
"""

def ask_llm_json(system_prompt: str, user_prompt: str) -> dict:
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]
    out = llm.invoke(messages).content

    # Extract first JSON object if the model adds extra text
    m = re.search(r"\{.*\}", out, flags=re.S)
    if not m:
        raise ValueError(f"LLM did not return JSON.\nRaw output:\n{out}")
    return json.loads(m.group(0))

C:\Users\nielsen.castelo\AppData\Local\Temp\ipykernel_11656\3038130106.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=MODEL, temperature=TEMPERATURE, base_url=BASE_URL)


# Data models + risk engine + tool simulators

In [3]:
from pydantic import BaseModel, Field

class PatientProfile(BaseModel):
    patient_id: str
    name: str
    age: int
    conditions: list[str] = Field(default_factory=list)
    meds: list[str] = Field(default_factory=list)

class CheckIn(BaseModel):
    timestamp: str
    symptoms: dict
    vitals: dict
    adherence: dict
    free_text: str = ""

def risk_engine(checkin: CheckIn, last_checkins: list[CheckIn]) -> tuple[str, list[str]]:
    """
    MVP rules-only risk engine.
    """
    s = checkin.symptoms or {}
    v = checkin.vitals or {}

    reasons = []
    urgent = False
    watch = False

    chest_pain = bool(s.get("chest_pain", False))
    severe_sob = s.get("shortness_of_breath", "") in ["severe", "at_rest"]
    syncope = bool(s.get("syncope", False))
    neuro = bool(s.get("stroke_symptoms", False))

    if chest_pain:
        urgent = True; reasons.append("Reported chest pain.")
    if severe_sob:
        urgent = True; reasons.append("Severe dyspnea or dyspnea at rest.")
    if syncope:
        urgent = True; reasons.append("Syncope reported.")
    if neuro:
        urgent = True; reasons.append("Acute neurological symptoms (possible stroke).")

    edema = bool(s.get("leg_swelling", False))
    sob_mod = s.get("shortness_of_breath", "") in ["mild", "moderate", "on_exertion"]

    weight = v.get("weight_kg", None)
    if weight is not None and last_checkins:
        prev_w = last_checkins[-1].vitals.get("weight_kg", None)
        if prev_w is not None and (weight - prev_w) >= 1.0:
            watch = True; reasons.append("Weight gain ≥ 1 kg vs prior day (possible fluid retention).")

    if edema:
        watch = True; reasons.append("Leg swelling/edema reported.")
    if sob_mod:
        watch = True; reasons.append("Mild/moderate dyspnea with exertion.")

    if urgent:
        return "urgent", reasons
    if watch:
        return "watch", reasons
    return "stable", ["No rule-based red flags detected today."]

# ---- Simulated tools (replace later with real integrations: EHR/FHIR, scheduling, messaging) ----
def tool_message_team(priority: str, summary: str, payload: dict):
    print(f"\n[TOOL] message_team | priority={priority}\n- summary: {summary}\n- payload: {json.dumps(payload, ensure_ascii=False)}")

def tool_schedule(kind: str, when: str, payload: dict):
    print(f"\n[TOOL] schedule | kind={kind} when={when}\n- payload: {json.dumps(payload, ensure_ascii=False)}")

def tool_self_care(tasks: list[str]):
    print("\n[TOOL] self_care tasks:")
    for t in tasks:
        print("-", t)

# LangGraph multi-agent flow (Patient Agent → Supervisor → Act)

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    profile: dict
    last_checkins: list
    checkin: dict
    rule_risk: str
    rule_reasons: list
    patient_agent_out: dict
    supervisor_out: dict

def node_rule_risk(state: AgentState) -> AgentState:
    c = CheckIn(**state["checkin"])
    last = [CheckIn(**x) for x in state.get("last_checkins", [])]
    risk, reasons = risk_engine(c, last)

    # computed / canonical features (avoid LLM hallucinations)
    delta_w = None
    if last:
        w_now = (c.vitals or {}).get("weight_kg", None)
        w_prev = (last[-1].vitals or {}).get("weight_kg", None)
        if w_now is not None and w_prev is not None:
            delta_w = float(w_now) - float(w_prev)

    state["rule_risk"] = risk
    state["rule_reasons"] = reasons
    state["delta_weight_kg"] = delta_w
    return state

def node_patient_agent(state: AgentState) -> AgentState:
    profile = PatientProfile(**state["profile"])
    c = state["checkin"]

    user_prompt = f"""\
        PATIENT:
        - Name: {profile.name}, Age: {profile.age}
        - Conditions: {profile.conditions}
        - Medications (list): {profile.meds}

        CHECK-IN TODAY (JSON):
        {json.dumps(c, ensure_ascii=False)}

        CANONICAL COMPUTED FEATURES (use these numbers exactly; do not guess):
        - delta_weight_kg={state.get("delta_weight_kg", None)}
        - rule_risk={state["rule_risk"]}
        - rule_reasons={state["rule_reasons"]}

        STRICT INSTRUCTIONS:
        - Do NOT compute or infer deltas; use delta_weight_kg above if present.
        - Do NOT invent any numeric value not present in the input JSON or computed features.

        Task:
        Return the Patient Agent JSON (schema required). Keep guidance safe.
        """
    out = ask_llm_json(PATIENT_AGENT_SYSTEM, user_prompt)
    state["patient_agent_out"] = out
    return state

def node_supervisor(state: AgentState) -> AgentState:
    user_prompt = f"""\
Patient Agent output (JSON):
{json.dumps(state["patient_agent_out"], ensure_ascii=False)}

Check-in data (JSON):
{json.dumps(state["checkin"], ensure_ascii=False)}

Rule-based risk:
{state["rule_risk"]} | {state["rule_reasons"]}

Decide per policies.
"""
    sup = ask_llm_json(SUPERVISOR_SYSTEM, user_prompt)
    state["supervisor_out"] = sup
    return state

def node_act(state: AgentState) -> AgentState:
    decision = state["supervisor_out"]["decision"]
    agent_out = state["patient_agent_out"]

    # If supervisor revised the output, use the safe revision
    if decision == "revise":
        agent_out = state["supervisor_out"]["safe_revision"]

    # --- HARD GATING (deterministic safety & ops) ---
    risk = agent_out.get("risk_level", "stable")

    filtered_actions = []
    for a in agent_out.get("proposed_actions", []):
        t = a.get("type", "none")
        payload = a.get("payload", {}) or {}

        # 1) Never notify the clinical team if stable (avoid spam)
        if risk == "stable" and t == "message_team":
            continue

        # 2) For urgent: prefer emergency escalation, not "schedule TBD"
        #    Allow schedule only if explicitly post-event follow-up with a real time
        if risk == "urgent" and t == "schedule":
            when = (payload.get("when") or "").strip()
            if when.lower() in ["", "tbd", "none", "null"]:
                continue

        # 3) Self-care must include non-empty tasks
        if t == "self_care":
            tasks = payload.get("tasks", [])
            if not isinstance(tasks, list) or len(tasks) == 0:
                continue

        filtered_actions.append(a)

    agent_out["proposed_actions"] = filtered_actions

    # --- Execute simulated tool actions ---
    for a in agent_out.get("proposed_actions", []):
        t = a.get("type", "none")
        payload = a.get("payload", {}) or {}

        if t == "message_team":
            priority = "urgent" if agent_out.get("risk_level") == "urgent" else "routine"
            tool_message_team(priority=priority, summary=agent_out.get("summary", ""), payload=payload)

        elif t == "schedule":
            tool_schedule(
                kind=payload.get("kind", "follow_up"),
                when=payload.get("when", "TBD"),
                payload=payload
            )

        elif t == "self_care":
            tool_self_care(payload.get("tasks", []))

    # --- Print safe patient-facing message ---
    print("\n=== SAFE PATIENT MESSAGE ===")
    print("Summary:", agent_out.get("summary", ""))
    print("Risk:", agent_out.get("risk_level", ""))
    print("Rationale:", " | ".join(agent_out.get("rationale", [])))

    if agent_out.get("questions"):
        print("\nQuick questions:")
        for q in agent_out["questions"]:
            print("-", q)

    if agent_out.get("patient_instructions"):
        print("\nInstructions:")
        for i in agent_out["patient_instructions"]:
            print("-", i)

    return state

def route_after_supervisor(state: AgentState) -> str:
    d = state["supervisor_out"]["decision"]
    if d == "block":
        return "end"
    return "act"

# Build graph
g = StateGraph(AgentState)
g.add_node("rule_risk", node_rule_risk)
g.add_node("patient_agent", node_patient_agent)
g.add_node("supervisor", node_supervisor)
g.add_node("act", node_act)

g.set_entry_point("rule_risk")
g.add_edge("rule_risk", "patient_agent")
g.add_edge("patient_agent", "supervisor")
g.add_conditional_edges("supervisor", route_after_supervisor, {"act": "act", "end": END})
g.add_edge("act", END)

app = g.compile()

# Demo: stable

In [5]:
profile = {
    "patient_id": "P001",
    "name": "Demo Patient",
    "age": 62,
    "conditions": ["Heart Failure", "Hypertension", "Post-MI"],
    "meds": ["beta-blocker (as prescribed)", "ACEi/ARB/ARNI (as prescribed)", "statin (as prescribed)"]
}

last_checkins = [
    {
        "timestamp": (datetime.now()-timedelta(days=1)).isoformat(),
        "symptoms": {"shortness_of_breath":"none", "leg_swelling": False, "chest_pain": False, "syncope": False, "stroke_symptoms": False},
        "vitals": {"weight_kg": 78.0, "bp_sys": 128, "bp_dia": 76, "hr": 68},
        "adherence": {"missed_doses": 0},
        "free_text": ""
    }
]

checkin_today = {
    "timestamp": datetime.now().isoformat(),
    "symptoms": {"shortness_of_breath":"none", "leg_swelling": False, "chest_pain": False, "syncope": False, "stroke_symptoms": False},
    "vitals": {"weight_kg": 78.2, "bp_sys": 130, "bp_dia": 78, "hr": 70},
    "adherence": {"missed_doses": 0},
    "free_text": "Feeling well today."
}

state = {
    "profile": profile,
    "last_checkins": last_checkins,
    "checkin": checkin_today,
    "rule_risk": "",
    "rule_reasons": [],
    "patient_agent_out": {},
    "supervisor_out": {}
}

_ = app.invoke(state)


=== SAFE PATIENT MESSAGE ===
Summary: The patient reported feeling well today with no symptoms of shortness of breath, leg swelling, chest pain, syncope, or stroke symptoms. Vitals are stable: weight is 78.2 kg, blood pressure is 130/78 mmHg, and heart rate is 70 bpm. No missed doses were reported.
Risk: stable
Rationale: No rule-based red flags detected today. | Vitals are within expected ranges for the patient's condition.

Quick questions:
- Have you noticed any changes in your diet or fluid intake?
- Are there any new symptoms that have not been mentioned?

Instructions:
- Continue taking medications as prescribed without missing doses.
- Monitor and record daily weight, blood pressure, and heart rate.


# Demo: watch (routine escalation)

In [ ]:
checkin_watch = {
    "timestamp": datetime.now().isoformat(),
    "symptoms": {"shortness_of_breath":"moderate", "leg_swelling": True, "chest_pain": False, "syncope": False, "stroke_symptoms": False},
    "vitals": {"weight_kg": 79.3, "bp_sys": 138, "bp_dia": 82, "hr": 78},
    "adherence": {"missed_doses": 1},
    "free_text": "More short of breath climbing stairs; legs look swollen."
}

state2 = dict(state)
state2["checkin"] = checkin_watch

_ = app.invoke(state2)

# Demo: urgent (urgent escalation)

In [ ]:
checkin_urgent = {
    "timestamp": datetime.now().isoformat(),
    "symptoms": {"shortness_of_breath":"at_rest", "leg_swelling": True, "chest_pain": True, "syncope": False, "stroke_symptoms": False},
    "vitals": {"weight_kg": 80.0, "bp_sys": 150, "bp_dia": 92, "hr": 98},
    "adherence": {"missed_doses": 0},
    "free_text": "Chest pain now and severe shortness of breath."
}

state3 = dict(state)
state3["checkin"] = checkin_urgent

_ = app.invoke(state3)


[TOOL] message_team | priority=urgent
- summary: Patient reports severe shortness of breath at rest, chest pain, and leg swelling with a weight of 80 kg. Blood pressure is elevated at 150/92 mmHg, heart rate is 98 bpm.
- payload: {"message": "Patient reports severe shortness of breath at rest, chest pain, and leg swelling. Immediate medical evaluation recommended."}

[TOOL] schedule | kind=follow_up when=TBD
- payload: {"action": "Arrange for immediate medical consultation or emergency services if symptoms persist or worsen."}

=== SAFE PATIENT MESSAGE ===
Summary: Patient reports severe shortness of breath at rest, chest pain, and leg swelling with a weight of 80 kg. Blood pressure is elevated at 150/92 mmHg, heart rate is 98 bpm.
Risk: urgent
Rationale: Reported chest pain. | Severe dyspnea or dyspnea at rest. | Leg swelling/edema reported.

Quick questions:
- Can you describe the nature of your chest pain (e.g., sharp, dull, pressure-like)?
- Have you noticed any changes in urine o